# Deep SRQ Network Architecture Ablation

Compares critic architectures while holding the rollout and SRE solver setup fixed:

- `joint_output`
- `per_agent_independent`
- `shared_trunk_separate_heads`

All runs use vectorized rollout with 32 environments and process-pool PATH SRE batching.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "bimatrix_game":
    BIMATRIX_DIR = ROOT
else:
    BIMATRIX_DIR = ROOT / "discrete_action_space" / "bimatrix_game"
DISCRETE_DIR = BIMATRIX_DIR.parent
for path in (BIMATRIX_DIR, DISCRETE_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from experiment_harness import BASE_SEED, configure_path_runtime
from vectorized_deep_srq import train_vectorized_deep_srq_experiment
from stats_utils import save_training_stats, summarize_rewards

In [ ]:
PATHWRAP = str(configure_path_runtime(DISCRETE_DIR))
OUTPUT_ROOT = BIMATRIX_DIR / "ablation_runs" / "network_architecture"

SCENARIOS = ("scenario1", "scenario2", "scenario3")
NETWORK_TYPES = (
    "joint_output",
    "per_agent_independent",
    "shared_trunk_separate_heads",
)

NUM_ENVS = 32
EPSILON = 0.5
EPSILON_SCHEDULE = "constant"
N_ENVIRONMENT_STEPS = 5_000
USE_GPU = False
SOLVER_NAME = "path_c_pool"

BASE_HP = {
    "sre_num_repeats": 4,
    "batch_size": 64,
    "learning_starts": 256,
    "sre_solver_workers": 4,
}

In [ ]:
results = {}
for scenario_index, scenario_key in enumerate(SCENARIOS):
    scenario_results = {}
    for network_index, network_type in enumerate(NETWORK_TYPES):
        seed = BASE_SEED + scenario_index * 1000 + network_index
        hp = BASE_HP | {"network_type": network_type}
        stats = train_vectorized_deep_srq_experiment(
            scenario_key=scenario_key,
            n_environment_steps=N_ENVIRONMENT_STEPS,
            num_envs=NUM_ENVS,
            epsilon_robust_initial=EPSILON,
            epsilon_schedule=EPSILON_SCHEDULE,
            seed=seed,
            pathwrap_path=PATHWRAP,
            solver_name=SOLVER_NAME,
            output_root=OUTPUT_ROOT / network_type,
            use_gpu=USE_GPU,
            write_plots=False,
            hyperparameter_overrides=hp,
            run_name_suffix=network_type,
        )
        stats["ablation_variant"] = network_type
        stats["ablation_mode"] = "network_architecture"
        scenario_results[network_type] = stats
    results[scenario_key] = scenario_results

save_training_stats(OUTPUT_ROOT / "network_architecture_manifest.txt", results)

In [ ]:
rows = []
for scenario_key, scenario_results in results.items():
    for network_type, stats in scenario_results.items():
        timing = stats["timing"]
        rewards = summarize_rewards(stats)
        env_steps = stats.get("total_environment_steps") or stats.get("n_environment_steps")
        rows.append({
            "scenario": scenario_key,
            "network_type": network_type,
            "wall_seconds": timing["wall_clock_seconds"],
            "env_steps": env_steps,
            "steps_per_second": env_steps / timing["wall_clock_seconds"],
            "sre_count": timing["sre_solve_time"]["count"],
            "mean_sre_ms": timing["sre_solve_time"]["mean_microseconds"] / 1000.0,
            "backend_count": timing["backend_solve_time"]["count"],
            "mean_backend_ms": timing["backend_solve_time"]["mean_microseconds"] / 1000.0,
            "agent1_mean_last": rewards[0]["MeanLastN"],
            "agent2_mean_last": rewards[1]["MeanLastN"],
        })

for row in rows:
    print(row)